# Chapter 3: Routing

**Estimated time: 15 minutes**

Your chunks are perfect. Your query rewriter is sharp. A user asks "What was Q3 revenue for the Pro segment?" and your pipeline searches a vector store of help articles. It finds nothing useful, because revenue does not live in help articles. It lives in a pandas table. Basic RAG never looks there.

In this notebook you will build a router. The router reads one question, decides which data source can answer it, and sends the question to that source. You will watch three very different questions flow through the same pipeline and land in three different places: the vector store, a quarterly revenue DataFrame, and an org chart. Same user interface, three different kinds of retrieval behind the scenes.

Run every cell top to bottom. When you reach the "Try it yourself" section, change the parameters and rerun.

## What you will see in three minutes

SkillAgents AI has three kinds of data.

1. **Unstructured docs** live in a FAISS vector store. Pricing PDF, refund policy PDF, product guide, billing FAQ, error codes, one outdated blog post. Six files. The kind of stuff a help article covers.
2. **Quarterly revenue** lives in a pandas DataFrame. One row per quarter per segment. Three segments: Student, Pro, Enterprise. Four quarters of 2025. Twelve rows. The kind of data you query in SQL.
3. **The org chart** lives in a small Python dict. Every manager maps to a list of their direct reports. The kind of data you traverse in a graph database.

You will ask three questions, one for each source.

- "What is the SkillAgents refund policy?" is a vector store question.
- "What was Q3 2025 revenue for the Pro segment?" is a revenue table question.
- "Who are the direct reports of Dan Alvarez?" is an org chart question.

First you will run the two non-vector questions through basic RAG, which sends every question to the vector store. Both answers will come back empty, because the information simply is not in the help articles. Then you will build an LLM router that reads each question and picks the right source. The same three questions get three clean answers.

After that you will build a semantic router that skips the LLM classification call and does the routing with a single embedding lookup. Faster. Cheaper. Slightly less flexible. You will see all three tradeoffs in action.

The most important thing to watch is not the exact words of the answer. It is the routing decision. A route of `revenue_table` means the pipeline knew to do a pandas lookup, not a semantic search. That is the full game of Chapter 3.

## Install the dependencies

Run the next cell once. In Colab it installs the Python packages fresh. Locally it assumes you already ran `pip install -r requirements.txt` in your virtual environment.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Cleaning up any prior chromadb install (silent)...")
    !pip uninstall -y -q chromadb chroma-hnswlib langchain-chroma 2>/dev/null

    print("Installing dependencies (takes about 60 seconds)...")
    !pip install -q \
        langchain==0.3.14 \
        langchain-openai==0.2.14 \
        langchain-community==0.3.14 \
        langchain-text-splitters==0.3.5 \
        faiss-cpu==1.13.2 \
        pypdf==5.1.0 \
        langsmith==0.2.6 \
        pandas==2.2.2 \
        matplotlib==3.9.4
    print("Done.")
else:
    print("Local environment detected. Skipping pip install.")
    print("Make sure you have run 'pip install -r requirements.txt' in your venv.")

In [ ]:
import os, sys

if IN_COLAB:
    # Always force a fresh clone so updates on GitHub are picked up. Without
    # this, a stale repo from a prior session would keep running the old utils.
    !rm -rf rag-for-pms
    !git clone -q https://github.com/DDRXV/rag-for-pms.git
    os.chdir("rag-for-pms")
else:
    # Local Jupyter: already inside the repo. Walk up to root if we are in chapters/.
    if os.path.basename(os.getcwd()) == "chapters":
        os.chdir("..")

# Python caches imported modules in sys.modules. Drop any cached utils.* so the
# next import reads the freshly cloned file, not a stale copy from an earlier
# session.
for cached in [m for m in sys.modules if m.startswith("utils")]:
    del sys.modules[cached]

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
from utils.shared import get_keys
get_keys()

## What you just did

`get_keys` pulled your OpenAI and LangSmith keys out of Colab secrets (you set these up in Chapter 1) and turned on LangSmith tracing for this notebook. Every LLM call, every retrieval call, and every router decision in this chapter gets logged automatically to a visual waterfall at [smith.langchain.com](https://smith.langchain.com). Open it in another tab after you run a few cells. You will be able to see the router decide which source to hit before any retrieval happens.

## Look at the three data sources

The vector store corpus is the same six documents from Chapter 1 and Chapter 2. The revenue table and the org chart are new. Both are Chapter 3 specific and live under `data/skillagents/ch3_*`.

Run the next cell to load all three sources and print a one-line summary of each. This is the same shape of loading you would do in production, except in production the revenue table would be a SQL connection and the org chart would be a Neo4j client.

In [ ]:
from utils.shared import load_corpus, load_revenue_table, load_org_chart

docs = load_corpus()
revenue_df = load_revenue_table()
org_pack = load_org_chart()

print()
print(f"Vector store corpus: {len(docs)} documents")
for d in docs:
    print(f"  {d.metadata.get('source', 'unknown'):25s}  {len(d.page_content):5d} chars")

print()
print(f"Revenue table: {revenue_df.shape[0]} rows, {revenue_df.shape[1]} columns")
print(f"  quarters: {sorted(revenue_df['quarter'].unique().tolist())}")
print(f"  segments: {sorted(revenue_df['segment'].unique().tolist())}")

print()
print(f"Org chart: {len(org_pack['graph'])} managers recorded")
for person in list(org_pack['graph'].keys())[:3]:
    print(f"  {person}")
print("  ...")

## Peek at what each source actually contains

Before any pipeline runs, look at the raw rows. The point is to see the answer sitting in exactly one place, so you know which source a router needs to pick.

In [ ]:
# The vector store has a refund policy clause in refund_policy.pdf.
refund = next(d for d in docs if d.metadata["source"] == "refund_policy.pdf")
refund_start = refund.page_content.find("Overview")
print("--- refund_policy.pdf, Overview section")
print(refund.page_content[refund_start:refund_start + 260])
print()

# The revenue table has exact Q3 2025 numbers per segment.
print("--- revenue table, Q3 2025 rows")
print(revenue_df[revenue_df["quarter"] == "Q3 2025"].to_string(index=False))
print()

# The org chart has Dan Alvarez and his three direct reports.
print("--- org chart, Dan Alvarez subtree")
dan = "Dan Alvarez (VP Engineering)"
print(f"{dan} ->")
for report in org_pack["direct_reports"](dan):
    print(f"  {report}")

Three sources. Three kinds of answers. A refund clause in prose. Three revenue rows with exact dollar amounts. Three named engineers reporting to one manager. No single index can answer all three questions, because no single index is the right shape.

## Build the vector index

The vector store is the same build you used in Chapter 1 and Chapter 2. `chunk_size=500`, `chunk_overlap=50`. Routing does not change the index. It only changes when the index gets used.

In [ ]:
from utils.shared import build_index

index = build_index()

## The problem: basic RAG sends everything to the vector store

Pretend your RAG system has no router. It sends every question through `search()` on the vector store. That is fine for the refund question, which lives in a help article. Watch what happens when you send the revenue and org questions the same way.

In [ ]:
from utils.shared import search, show_results, generate_answer

question_revenue = "What was Q3 2025 revenue for the Pro segment?"
results = search(index, question_revenue, k=3)
show_results(results, question_revenue)

In [ ]:
# Feed the top chunks to the LLM and see what it answers.
basic_answer_revenue = generate_answer(results, question_revenue)
print("Q:", question_revenue)
print()
print("A:", basic_answer_revenue)

Basic RAG retrieved three chunks from the pricing PDF and the product guide. None of them mention Q3 2025 revenue, because that number does not exist in the corpus. It exists in the revenue table. The LLM politely tells you the context does not contain the answer. That is the honest outcome. The dangerous outcome would be an LLM that hallucinates a number to fill the gap.

In [ ]:
question_org = "Who are the direct reports of Dan Alvarez?"
results_org = search(index, question_org, k=3)
basic_answer_org = generate_answer(results_org, question_org)
print("Q:", question_org)
print()
print("A:", basic_answer_org)

Same story. The org chart is a Python dict. The vector store is a pile of help articles. No amount of fancy chunking or query rewriting will put a direct reports list into a refund policy PDF.

Basic RAG is not broken. It is under-provisioned. It only knows about one source, and two of your three questions need a different source. You need a router.

## LLM router: classify the question, then retrieve

The LLM router is the most common approach in production. The idea is simple. Before retrieval, you send the question to a small LLM with a system prompt that says "classify this question into one of these categories". The LLM returns a category. Your code then dispatches to the right retrieval function.

`classify_route_llm` in `utils/shared.py` does exactly that. It takes a question and a dict of routes. It returns the winning route name as a plain string.

In [ ]:
from utils.shared import classify_route_llm, DEFAULT_CH3_ROUTES

# Print the routes the classifier will choose from.
for name, description in DEFAULT_CH3_ROUTES.items():
    print(f"- {name}")
    print(f"    {description}")
    print()

Read the three route descriptions above. The descriptions are the entire prompt that tells the LLM what each route is for. Short and concrete beats clever and vague. A classifier prompt that says "a vector store of knowledge" will drift. A classifier prompt that says "questions about refund terms, pricing plans, product behavior, or error codes" will not.

Now send each of the three questions through the classifier.

In [ ]:
product_context = (
    "SkillAgents AI, a cohort-based course platform. Three data sources: "
    "a vector store of help documentation (pricing, refund policy, product "
    "guide, billing FAQ, error codes), a pandas table of quarterly revenue "
    "by Student, Pro, and Enterprise segments, and an in-memory org chart "
    "dict mapping managers to direct reports."
)

questions = [
    "What is the SkillAgents refund policy?",
    "What was Q3 2025 revenue for the Pro segment?",
    "Who are the direct reports of Dan Alvarez?",
]

for q in questions:
    route = classify_route_llm(q, context=product_context)
    print(f"{route:15s}  <-  {q}")

Three different questions. Three different routes. The classifier did not read any document. It did not run any retrieval. It only looked at the shape of the question and the route descriptions. That is the fast, cheap part of routing.

Now run the full pipeline. Classify, dispatch to the right source, generate the final answer. `route_and_answer` glues those three steps together.

In [ ]:
from utils.shared import route_and_answer

context_pack = {
    "index": index,
    "revenue_df": revenue_df,
    "org_pack": org_pack,
}

outcomes = []
for q in questions:
    out = route_and_answer(q, context_pack, context=product_context)
    outcomes.append(out)
    print("---")
    print("Q:      ", q)
    print("Route:  ", out["route"])
    print("Answer: ", out["answer"])
    print()

Three questions. Three clean answers. Compare the revenue and org answers to the basic RAG answers you saw a few cells ago. Basic RAG said "the context does not contain the answer" for both. The routed pipeline returned exact numbers for Q3 2025 Pro revenue and the exact list of direct reports for Dan Alvarez.

That is the lesson. Retrieval quality is not only about good embeddings. It is about asking the right index in the first place.

## Look at the raw output from each route

Answers are the payoff, but a PM reviewing a RAG system needs to read the raw output too. A good route can still fail if the source returns the wrong rows. Run the cell below to see what each route actually produced.

In [ ]:
# The vector route returns FAISS chunks.
vector_out = outcomes[0]["route_output"]
print(f"Route: {vector_out['route']}")
show_results(vector_out["detail"], questions[0])

In [ ]:
# The revenue route returns a filtered DataFrame plus an LLM-generated plan.
revenue_out = outcomes[1]["route_output"]["detail"]
print(f"LLM plan for the revenue question:")
print(f"  {revenue_out['plan']}")
print()
print("Filtered rows:")
print(revenue_out["rows"].to_string(index=False))
print()
print(f"Summary: {revenue_out['summary']}")

The revenue route did two steps. First the LLM turned "Q3 2025 revenue for the Pro segment" into a JSON plan that names the quarters, segments, metric, and aggregation. Then Python filtered the DataFrame with that plan. No LLM call touched the data values themselves. The LLM only wrote the query. The values came from pandas. That is the pattern you want for any numerical source: LLM writes the query, the database executes the query, the LLM only sees the result.

In [ ]:
# The org chart route returns the matched manager and the direct reports list.
org_out = outcomes[2]["route_output"]["detail"]
print(f"Matched manager: {org_out['person']}")
print("Direct reports:")
for r in org_out["reports"]:
    print(f"  {r}")

## Semantic router: skip the LLM classification call

The LLM router works, but every question costs one extra LLM call before retrieval even starts. For a high-traffic RAG system that is real money and real latency. The semantic router is the alternative.

The idea: for each route, write a few example phrases by hand. Embed every example once, at startup. When a question arrives, embed the question and compute cosine similarity against every pre-computed example. The highest score picks the route.

One embedding call per question. No LLM call. Sub-millisecond arithmetic after that.

In [ ]:
from utils.shared import build_semantic_router, semantic_route

route_examples = {
    "vector_store": [
        "What is the refund policy?",
        "Explain the pricing plans",
        "How does a cohort work?",
        "What does error code E-4012 mean?",
    ],
    "revenue_table": [
        "What was revenue last quarter?",
        "Total sales by segment",
        "How many paying customers in Q2?",
        "Average revenue per segment",
    ],
    "org_chart": [
        "Who reports to the VP of Engineering?",
        "What is the team structure?",
        "List the direct reports of the CEO",
        "Who manages the ML team?",
    ],
}

semantic_bundle = build_semantic_router(route_examples)
print(
    f"Pre-computed {len(semantic_bundle['labels'])} example embeddings across "
    f"{len(semantic_bundle['routes'])} routes."
)

Twelve example phrases, embedded once. That is the entire setup cost. Now route the three questions and watch which route wins each one.

In [ ]:
from utils.shared import show_route_scores

for q in questions:
    out = semantic_route(semantic_bundle, q)
    print(f"Q: {q}")
    print(f"  winner: {out['winner']}")
    print(f"  scores:")
    for route, score in sorted(out['scores'].items(), key=lambda kv: -kv[1]):
        print(f"    {route:15s}  {score:.3f}")
    print()

Three questions. Three correct routes. No LLM call. The winning route scored around 0.5 to 0.6 on cosine similarity, while the runner up routes scored around 0.2 to 0.3. That is a strong gap. When the gap is narrow, the semantic router is guessing. When the gap is wide, it is confident.

This is where the tradeoffs bite. The LLM router can handle edge cases because it reasons about intent. The semantic router can only match on pattern similarity. A question that does not look like any pre-defined example will misroute. You can fix this by adding more examples per route, but now you are curating a training set. The semantic router is fast and cheap and requires ongoing maintenance. The LLM router is slow and expensive and requires almost none.

## Multi-source routing: one question, two data sources

Some questions need data from more than one place at the same time. Imagine a product manager types this into the SkillAgents internal assistant:

> Compare our refund policy terms against Q3 2025 Pro segment revenue, and tell me how much money is at risk if every Pro subscriber requested a refund today.

A single-source router would have to pick one or the other. Picking the refund policy drops the dollar amount. Picking the revenue table drops the policy clauses. Either way the answer is incomplete.

A multi-source router classifies the question by **intents**, not destinations. One question can have several intents. Each intent maps to one source. You run the sources in parallel and feed everything to the LLM together.

Build a tiny version of this by hand. Decompose the question by hand into two clear sub-intents, route each one, and merge the results.

In [ ]:
compound_question = (
    "Compare the SkillAgents refund policy against Q3 2025 revenue for the "
    "Pro segment, and tell me how much money is at risk if every Pro "
    "subscriber requested a refund today."
)

sub_intents = [
    ("vector_store", "What are the SkillAgents refund policy terms for the Pro Annual plan?"),
    ("revenue_table", "What was Q3 2025 revenue for the Pro segment?"),
]

from utils.shared import run_route

multi_context_blocks = []
for route_name, sub_q in sub_intents:
    out = run_route(route_name, sub_q, context_pack)
    print(f"--- {route_name}: {sub_q}")
    print(out["result_text"][:500])
    print()
    block = "Source: " + route_name + "\n" + out["result_text"]
    multi_context_blocks.append(block)

In [ ]:
# Feed both blocks into the LLM together and let it synthesize the answer.
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

system = (
    "You answer a compound business question using context drawn from "
    "multiple sources. Cite the exact numbers and policy clauses you used. "
    "If the context does not contain enough information for any part of "
    "the answer, say so plainly."
)
combined_context = "\n\n===\n\n".join(multi_context_blocks)
user = (
    f"Context:\n{combined_context}\n\n"
    f"Question: {compound_question}"
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
multi_answer = llm.invoke(
    [SystemMessage(content=system), HumanMessage(content=user)]
).content.strip()
print(multi_answer)

Read the answer above. It quotes the refund clause (where the two months of service used rule lives) and the exact dollar number from the revenue table ($892,000 for Q3 2025 Pro). The LLM did not invent either value. Both came from the right source. The LLM only synthesized them.

The key insight from this cell: classify intents, not destinations. A single question can have two intents that map to two sources. The router is a decomposition step, not a single classification step. In production the decomposition itself is usually an LLM call that returns a list of (route, sub_question) pairs. You just did that step by hand. The shape is identical.

## Decision framework: which router, when

You have three routers now. Here is how to pick.

| Router | Use when | Skip when |
|---|---|---|
| **LLM router** | Your questions are diverse and unpredictable. Edge cases matter. You can afford one extra LLM call per question. | Your traffic is high enough that one extra LLM call per question is too slow or too expensive. |
| **Semantic router** | Your question patterns are well defined and repeatable. You need sub-millisecond routing. Cost per query matters. | Questions are too open ended to cover with a fixed set of example phrases. |
| **Multi-source routing** | Questions often need data from more than one source. A single route produces incomplete answers. | Every question fits cleanly in one source. |

The progression most teams follow:

1. One data source. No router needed. Direct vector search is enough.
2. Two or three data sources. Add an LLM router. Ship it. Measure how often it misroutes.
3. High traffic. Switch the routing layer to a semantic router, keep the LLM router as a fallback for low confidence cases.
4. Compound questions show up in the logs. Add a multi-source decomposition step.

Do not skip steps. Do not add multi-source routing before you have observed compound questions in real traffic. Do not switch to a semantic router before you have measured LLM router latency in production. Every router adds complexity. Add complexity only where it pays for itself.

## Open your LangSmith trace

Open [smith.langchain.com](https://smith.langchain.com), click into the project called `rag-for-pms`, and open the most recent trace. You will see one LLM call for the classifier, one LLM call for the pandas plan generator, one LLM call for the org chart person picker, and one LLM call for the final answer. Every step is visible. Every prompt is visible. Five minutes of clicking around the trace view will teach you more about how routing actually runs than an hour of reading this notebook.

## What you can do at work on Monday

Routing is the second biggest fix you can make to a production RAG system after query translation. Here are the three questions to ask on your next RAG review.

1. How many distinct data sources does your RAG system actually have? If the answer is "just the vector store" but your users ask about numbers, relationships, or anything structured, you are shipping hallucinations. Add at least one more source.
2. When a user asks a question that does not belong in the vector store, what happens? Does the pipeline route it somewhere else, or does it search the vector store anyway and hope? "It searches anyway" is the single most common failure mode for companies that stopped at Chapter 1 of the RAG journey.
3. Has anyone measured the cost of the classifier call on high traffic queries? If the answer is "we have not looked", you either have headroom you are not using or a latency bill you do not know about. Five minutes of LangSmith trace inspection answers this cleanly.

A team that cannot answer these three questions is probably running a one-source RAG system and calling it a router.

## Try it yourself

Pick any of these. Change the value in the relevant cell, rerun, watch what happens.

1. **Add a fourth route.** The SkillAgents team ships a support ticketing API. In `DEFAULT_CH3_ROUTES` (open `utils/shared.py` or copy the dict into a new cell), add a fourth entry called `support_api` with a short description like "questions about a specific support ticket status, a customer escalation, or a known issue tracked in the internal ticket system". Rerun `classify_route_llm` with the question "What is the status of ticket T-1042 escalation?". The classifier should pick `support_api`. You did not have to write any retrieval code. Adding a route description is enough to extend the router.

2. **Weaken the classifier prompt.** Pass `context=None` to `classify_route_llm` for the revenue question. Without the product context block, does the classifier still pick `revenue_table`, or does it drift? This is the exact failure mode you saw in Chapter 2 with `generate_query_variants`. Classifier prompts, like rewriter prompts, need the product context to stay grounded.

3. **Break the semantic router with a new question shape.** Run `semantic_route` with the question "How much does the Student plan cost, and how does it compare to Pro?". This is a vector store question that sounds a little like a revenue question. Watch the scores. Is the winner still `vector_store`, and by how much? Narrow gaps are how semantic routers misroute. Add one more example phrase under `vector_store` that mentions "plan cost" and rerun. The gap should widen.

## Homework

Two exercises you can do on your own. Each takes about fifteen minutes.

1. **Build a full semantic router for your own product.** Pick three data sources from a real product at your company. For each source, write five example phrases the way a user would actually type them. Paste them into `build_semantic_router`. Then draft ten real user questions and route each one. How many land on the source you expected? The miss rate is the gap between your example phrases and the shape of real user questions. Narrow that gap by rewriting the examples, not the questions.

2. **Measure the cost of routing.** Time the LLM router and the semantic router on the same batch of twenty questions. Record the total latency and the approximate token cost for each one. The semantic router should be roughly ten to fifty times faster per query, and the cost gap is even bigger because semantic routing only uses one embedding call per question instead of a full chat completion. Write a one-paragraph recommendation about which router you would ship first and why. Bring the paragraph to your next RAG review.

## What is next

In Chapter 4 your router picks the right source, and your pipeline retrieves from that source, but the source itself is a big pile of unstructured documents that share a lot of vocabulary. A user asks "What does error code E-4012 mean?" and your vector search pulls in six chunks about different error codes because the word "error" shows up everywhere. Pure semantic similarity is not enough. You need keyword matching plus semantic matching, merged into one ranked list.

Chapter 4 is about hybrid search. You will see how BM25 and vector similarity work together, and why every production RAG system uses both. See you there.